In [1]:
import json
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import operator
from statistics import mean
from collections import Counter

In [2]:
with open('data/train-claims.json', 'r') as input_file:
    train_claim_data = json.load(input_file)

# Read in development data (claim)
with open('data/dev-claims.json', 'r') as input_file:
    dev_claim_data = json.load(input_file)

# Read in test data (claim)
with open('data/test-claims-unlabelled.json', 'r') as input_file:
    test_claim_data = json.load(input_file)

# Read in evidence data
with open('data/evidence.json', 'r') as input_file:
    evi_data = json.load(input_file)

train_claim_id = list(train_claim_data.keys())
train_claim_text  = [ v["claim_text"] for v in train_claim_data.values()]
dev_claim_text  = [ v["claim_text"] for v in dev_claim_data.values()]


In [3]:
import json
import nltk
from nltk.corpus import stopwords
import re

# 下载 NLTK 数据
nltk.download('punkt')

# 从文件中读取原始的evidence数据
with open('data/evidence.json', 'r') as input_file:
    evi_data = json.load(input_file)

# 预处理去除非英语证据
english_evidence = {}  # 用于存储经过预处理后的英语证据

for evi_id, evi_text in evi_data.items():
    # 判断文本是否为英语
    words = nltk.word_tokenize(evi_text)
    english_words = [word for word in words if word.isalpha()]
    if len(english_words) / len(words) > 0.5:  # 如果超过一半的词是英文单词，则认为是英语文本
        # 如果是英语，则进行进一步的预处理，例如去除停用词等
        english_text = ' '.join(english_words)
        english_evidence[evi_id] = english_text

# 输出筛选后的evidence数量
filtered_evidence_count = len(english_evidence)
print("Filtered Evidence Count:", filtered_evidence_count)



# 提取训练集和开发集中出现频率最高的名词或者字母与数字的组合
train_claim_text = [claim_value['claim_text'] for claim_value in train_claim_data.values()]
dev_claim_text = [claim_value['claim_text'] for claim_value in dev_claim_data.values()]

# 定义停用词集合
stop_words = set(stopwords.words('english'))
# 提取名词或者字母与数字的组合
claim_words = []
for text_list in [train_claim_text, dev_claim_text]:
    for text in text_list:
        tokens = nltk.word_tokenize(text)
        for token in tokens:
            # 只保留名词或者字母与数字的组合，并且不在停用词集合中的单词
            if re.match('^[a-zA-Z0-9]+$', token) and token.lower() not in stop_words and nltk.pos_tag([token])[0][1] in ['NN', 'NNS', 'NNP', 'NNPS']:
                claim_words.append(token.lower())

# 统计出现频率最高的单词
top_words_counter = Counter(claim_words)
# 获取出现频率最高的单词列表，最多添加500个单词
top_words = [word for word, _ in top_words_counter.most_common(100)]

# 打印出现频率最高的单词列表
print("Top Words:", top_words)

# 创建一个新的 dictionary 用于存储符合条件的 evidence
climate_dic = {}

# 遍历英语证据，将包含出现频率最高的单词的 evidence 收集到 climate_dic 中
for evi_id, evi_text in english_evidence.items():
    words = nltk.word_tokenize(evi_text)
    if any(word.lower() in top_words for word in words):
        climate_dic[evi_id] = evi_text

# 输出 climate_dic 中的 evidence 个数
print("Number of evidence in climate_dic:", len(climate_dic))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\XZH\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Filtered Evidence Count: 1181638
Top Words: ['climate', 'co2', 'ice', 'change', 'temperature', 'sea', 'carbon', 'years', 'temperatures', 'scientists', 'emissions', 'rise', 'earth', 'level', 'dioxide', 'greenhouse', 'record', 'past', 'ocean', 'year', 'levels', 'ipcc', 'century', 'data', 'planet', 'evidence', 'world', 'increase', 'energy', 'trend', 'heat', 'surface', 'effect', 'models', 'water', 'solar', 'human', 'gas', 'humans', 'weather', 'show', 'time', 'degrees', 'warmer', 'cause', 'changes', 'extreme', 'decades', 'found', 'greenland', 'events', 'antarctica', 'today', 'times', 'amount', 'period', 'sun', 'gases', 'polar', 'percent', 'study', 'cent', 'satellite', 'measurements', 'report', 'research', 'impact', 'oceans', 'half', 'united', 'rate', 'shows', 'states', 'cold', 'science', 'activity', 'fact', 'air', 'mean', 'summer', 'studies', 'warm', 'fossil', 'cycle', 'glaciers', 'stations', 'increases', 'age', 'australia', 'el', 'land', 'means', 'forests', 'term', 'trends', 'decline', 'ce